* usage log 확인

In [1]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더(hanwha-agent)를 찾아 루트로 삼는다
# - 폴더 이름(day02)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)                                  # 상대경로(.env 등)의 기준
SANDBOX = ROOT / "sandbox" / "w4" / "day02"

# app 패키지를 import 할 수 있게 backend 를 모듈 검색 경로 맨 앞에 넣는다
# - os.chdir 만으로는 import 경로가 바뀌지 않는다
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

print("프로젝트 루트  :", ROOT)

프로젝트 루트  : C:\workspace\hanwha-agent


In [2]:
import json

from sqlalchemy import func, select

import app.integrations.factory as factory
from app.db.session import session_scope
from app.integrations.ports import LLMResult
from app.models import UsageLog
from app.services import chat_service

class StubLLM:
    name = "stub"

    def __init__(self, replies: list[str]) -> None:
        self.replies = list(replies)
        self.calls = 0

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        text = self.replies[min(self.calls, len(self.replies) - 1)]
        self.calls += 1
        return LLMResult(
            text=text,
            model="claude-haiku-4-5",
            input_tok=1200,
            output_tok=300,
            cost_krw=3.8,    
            latency_ms=900,
        )

GOOD = json.dumps(
    {
        "answer": "부산 출장 숙박비는 1박 7만원까지 지원됩니다.",
        "sources": [
            {
                "doc_id": "DOC-HR-014",
                "title": "국내출장 여비 규정",
                "version": "v2.0",
                "locator": "제12조",
            }
        ],
        "enough_evidence": True,
    },
    ensure_ascii=False,
)

BAD = json.dumps(
    {"answer": "부산 출장 숙박비는 1박 7만원까지 지원됩니다.", "enough_evidence": True},
    ensure_ascii=False,
)

def rows_for(run_id: str) -> int:
    with session_scope() as session:
        return session.scalar(
            select(func.count()).select_from(UsageLog).where(UsageLog.run_id == run_id)
        )


print("재료 준비 완료 — StubLLM  GOOD  BAD  rows_for()")

재료 준비 완료 — StubLLM  GOOD  BAD  rows_for()


- 재시도 한 번 = 과금 한 번

In [3]:
# 재시도가 usage_logs 에 그대로 남는지 확인한다
# - BAD 는 sources 가 빠진 응답이라 스키마 검증에서 걸리고, 다음 시도로 넘어간다
# - 그 실패한 호출의 토큰도 이미 과금됐으므로 usage 행이 함께 쌓여야 한다
original = factory.get_llm
CASES = [
    ("① 한 번에 성공       ", [GOOD]),
    ("② 한 번 실패 후 성공 ", [BAD, GOOD]),
]

for label, replies in CASES:
    try:
        factory.get_llm = lambda r=replies: StubLLM(r)   # r=replies : 늦은 바인딩을 피한다
        out = chat_service.ask(question="부산 출장 숙박비 한도가 얼마인가요?")
    finally:
        factory.get_llm = original

    print(f"{label} {out.run_id}  attempts={out.attempts}  usage_logs {rows_for(out.run_id)}건")

print()
print("attempts 와 usage_logs 행 수가 같아야 한다 — 다르면 기록 위치가 잘못된 것이다")

① 한 번에 성공        RUN-8882  attempts=1  usage_logs 1건
17:14:23 WARNING  app.services.chat_service 스키마 위반 1/3회 : sources: Field required
② 한 번 실패 후 성공  RUN-8883  attempts=2  usage_logs 2건

attempts 와 usage_logs 행 수가 같아야 한다 — 다르면 기록 위치가 잘못된 것이다


- 사람별·모델별 집계

In [4]:
# 사용량 집계 
from sqlalchemy import func, select 

from app.db.session import session_scope
from app.models import Run, UsageLog, User

with session_scope() as session:
    per_person = session.execute(
        select(
            User.name, 
            User.emp_no, 
            func.count(UsageLog.id), 
            func.sum(UsageLog.cost_krw), 
        )
        .join(Run, UsageLog.run_id == Run.id)
        .join(User, Run.user_id == User.id)
        .group_by(User.name, User.emp_no)
    ).all() 

    print("***사람별 결과***")
    for name, emp_no, count, total in per_person:
        print(f" {name} ({emp_no}) : {count}건 {round(total or 0, 1)}원")

    in_tok, out_tok = session.execute(
        select(func.sum(UsageLog.input_tok), func.sum(UsageLog.output_tok))
    ).one()
    in_tok, out_tok = in_tok or 0, out_tok or 0 
    share = out_tok / (in_tok + out_tok) * 100 if (in_tok + out_tok) else 0.0

    print("***토큰 비중***")
    print(f" 입력 {in_tok}, 출력 {out_tok}")
    print(f" 출력 비중 : {share:.1f} %")


***사람별 결과***
 김민준 (2019-0412) : 51건 135.3원
***토큰 비중***
 입력 61200, 출력 15300
 출력 비중 : 20.0 %
